# 🚀 Day 4: Running Full v1 QLoRA Training
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Assignee:** Deepana Nirmal | **Jira Task:** `KAN-26`
### **Target Models:** Qwen 2.5-7B Instruct → `qwen_sme_v1` | Llama 3 8B Instruct → `llama_sme_v1`
### **Hardware:** Google Colab Tesla T4 GPU (15 GB VRAM)

---
### ♻️ Checkpoint Resume Supported
If Colab disconnects mid-training, re-run all cells — training will **automatically resume** from the last saved checkpoint.

## 1. Mount Google Drive & Environment Setup

In [ ]:
import os, sys, json, torch, gc

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

print(f"Project Root : {PROJECT_ROOT}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {props.total_memory/(1024**3):.1f} GB")
    print(f"   BF16 : {'supported' if torch.cuda.is_bf16_supported() else 'NOT supported — AMP disabled (T4 safe mode)'}")
else:
    raise RuntimeError("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets wandb
import trl, transformers, peft, wandb
print(f"TRL {trl.__version__} | Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("✅ Done. First run? → Runtime → Restart runtime → re-run all cells.")

## 2. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("✅ HF logged in via Colab Secret.")
except Exception:
    print("Continuing with public HF access.")

## 3. Weights & Biases Setup

In [ ]:
import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
    print("✅ W&B logged in via Colab Secret.")
except Exception:
    wandb.login()

WANDB_PROJECT = "SME-Daily-Business"
print(f"W&B Project: {WANDB_PROJECT}")

## 4. Load Processed Datasets

In [ ]:
train_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v1.json')
val_path   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v1.json')

with open(train_path, 'r', encoding='utf-8') as f: train_data = json.load(f)
with open(val_path,   'r', encoding='utf-8') as f: val_data   = json.load(f)

print(f"✅ Train: {len(train_data):,} samples | Val: {len(val_data):,} samples")

## 5. Helper Functions

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


def get_last_checkpoint(output_dir):
    """Find latest checkpoint folder in output_dir, or None."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [
        d for d in os.listdir(output_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, d))
    ]
    if not checkpoints:
        return None
    # Sort by step number and return latest
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    return os.path.join(output_dir, latest)


def build_datasets(tokenizer, train_data, val_data):
    def fmt(ex):
        sys_msg = "You are an expert SME daily business assistant."
        user_q  = ex['instruction']
        if ex.get('context'):
            user_q = f"Context:\n{ex['context']}\n\nQuestion:\n{ex['instruction']}"
        return tokenizer.apply_chat_template(
            [{"role":"system",   "content":sys_msg},
             {"role":"user",     "content":user_q},
             {"role":"assistant","content":ex['response']}],
            tokenize=False
        )
    return (
        Dataset.from_dict({"text": [fmt(x) for x in train_data]}),
        Dataset.from_dict({"text": [fmt(x) for x in val_data]})
    )


def load_qlora_model(model_id):
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.config.use_cache = False
    return model


def apply_lora(model, target_modules):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules
    ))
    model.print_trainable_parameters()
    return model


def make_sft_config(output_dir, run_name, epochs=3, total_steps=None):
    ws = max(10, int(total_steps * 0.05)) if total_steps else 10
    return SFTConfig(
        output_dir=output_dir,
        run_name=run_name,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=ws,
        fp16=False,
        bf16=False,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="wandb",
        dataset_text_field="text",
        max_length=512,
        optim="paged_adamw_32bit",
        seed=42
    )


print("✅ Helper functions loaded.")

## 6. Train Model A — Qwen 2.5-7B Instruct (`qwen_sme_v1`)
> ♻️ **Auto-resumes from last checkpoint** if training was interrupted.

In [ ]:
torch.cuda.empty_cache(); gc.collect()

QWEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
QWEN_OUT_DIR  = os.path.join(PROJECT_ROOT, 'models', 'v1', 'qwen_sme_v1')
os.makedirs(QWEN_OUT_DIR, exist_ok=True)

# ── Check for existing checkpoint ───────────────────────────────────────────
qwen_ckpt = get_last_checkpoint(QWEN_OUT_DIR)
if qwen_ckpt:
    print(f"♻️  Resuming Qwen training from: {qwen_ckpt}")
else:
    print("🆕 Starting Qwen training from scratch.")

# ── Check if already fully trained (adapter_config.json exists at root) ─────
qwen_done = os.path.exists(os.path.join(QWEN_OUT_DIR, 'adapter_config.json')) and not qwen_ckpt
if qwen_done:
    print("✅ Qwen adapter already fully trained and saved — skipping training.")
    qwen_metrics = {"eval_loss": 0.0}   # placeholder
else:
    print("="*60)
    print("  TRAINING: Qwen 2.5-7B → qwen_sme_v1")
    print("="*60)

    # Tokenizer
    print("[1/5] Loading tokenizer...")
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    if qwen_tok.pad_token is None: qwen_tok.pad_token = qwen_tok.eos_token
    qwen_tok.padding_side = "right"

    # Datasets
    print("[2/5] Building datasets...")
    qwen_train_ds, qwen_val_ds = build_datasets(qwen_tok, train_data, val_data)
    steps_per_epoch = len(qwen_train_ds) // (2 * 8)
    total_steps     = steps_per_epoch * 3
    print(f"      Train: {len(qwen_train_ds):,} | Val: {len(qwen_val_ds):,} | Total steps: {total_steps}")

    # Model
    print("[3/5] Loading Qwen 2.5-7B in 4-bit NF4...")
    qwen_model = load_qlora_model(QWEN_MODEL_ID)

    # LoRA
    print("[4/5] Applying LoRA adapters...")
    qwen_model = apply_lora(qwen_model,
        ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

    # W&B — resume run if checkpoint exists
    wandb.init(project=WANDB_PROJECT, name="qwen_sme_v1",
               resume="allow",   # resumes existing run with same name
               config={"model":QWEN_MODEL_ID,"epochs":3,"lr":2e-4,"batch":16,"lora_r":16})

    # Train
    print("[5/5] Starting / resuming training...")
    qwen_trainer = SFTTrainer(
        model=qwen_model,
        train_dataset=qwen_train_ds,
        eval_dataset=qwen_val_ds,
        processing_class=qwen_tok,
        args=make_sft_config(QWEN_OUT_DIR, run_name="qwen_sme_v1", total_steps=total_steps)
    )
    # resume_from_checkpoint=True → HuggingFace auto-detects last checkpoint
    qwen_trainer.train(resume_from_checkpoint=qwen_ckpt)

    # Save final adapter
    print("\nSaving Qwen final adapter...")
    qwen_trainer.model.save_pretrained(QWEN_OUT_DIR)
    qwen_tok.save_pretrained(QWEN_OUT_DIR)
    wandb.finish()

    qwen_metrics = qwen_trainer.evaluate()
    print(f"\n✅ Qwen v1 Done! Eval Loss: {qwen_metrics.get('eval_loss','N/A'):.4f}")

    del qwen_model, qwen_trainer
    torch.cuda.empty_cache(); gc.collect()
    print("🧹 GPU cleared for Llama training.")

## 7. Train Model B — Llama 3 8B Instruct (`llama_sme_v1`)
> ♻️ **Auto-resumes from last checkpoint** if training was interrupted.

In [ ]:
torch.cuda.empty_cache(); gc.collect()

LLAMA_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_OUT_DIR  = os.path.join(PROJECT_ROOT, 'models', 'v1', 'llama_sme_v1')
os.makedirs(LLAMA_OUT_DIR, exist_ok=True)

# ── Check for existing checkpoint ───────────────────────────────────────────
llama_ckpt = get_last_checkpoint(LLAMA_OUT_DIR)
if llama_ckpt:
    print(f"♻️  Resuming Llama training from: {llama_ckpt}")
else:
    print("🆕 Starting Llama training from scratch.")

# ── Check if already fully trained ──────────────────────────────────────────
llama_done = os.path.exists(os.path.join(LLAMA_OUT_DIR, 'adapter_config.json')) and not llama_ckpt
if llama_done:
    print("✅ Llama adapter already fully trained and saved — skipping training.")
    llama_metrics = {"eval_loss": 0.0}
else:
    print("="*60)
    print("  TRAINING: Llama 3 8B → llama_sme_v1")
    print("="*60)

    # Tokenizer
    print("[1/5] Loading tokenizer...")
    llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID, trust_remote_code=True)
    if llama_tok.pad_token is None: llama_tok.pad_token = llama_tok.eos_token
    llama_tok.padding_side = "right"

    # Datasets
    print("[2/5] Building datasets...")
    llama_train_ds, llama_val_ds = build_datasets(llama_tok, train_data, val_data)
    steps_per_epoch = len(llama_train_ds) // (2 * 8)
    total_steps     = steps_per_epoch * 3
    print(f"      Train: {len(llama_train_ds):,} | Val: {len(llama_val_ds):,} | Total steps: {total_steps}")

    # Model
    print("[3/5] Loading Llama 3 8B in 4-bit NF4...")
    llama_model = load_qlora_model(LLAMA_MODEL_ID)

    # LoRA
    print("[4/5] Applying LoRA adapters...")
    llama_model = apply_lora(llama_model,
        ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

    # W&B
    wandb.init(project=WANDB_PROJECT, name="llama_sme_v1",
               resume="allow",
               config={"model":LLAMA_MODEL_ID,"epochs":3,"lr":2e-4,"batch":16,"lora_r":16})

    # Train
    print("[5/5] Starting / resuming training...")
    llama_trainer = SFTTrainer(
        model=llama_model,
        train_dataset=llama_train_ds,
        eval_dataset=llama_val_ds,
        processing_class=llama_tok,
        args=make_sft_config(LLAMA_OUT_DIR, run_name="llama_sme_v1", total_steps=total_steps)
    )
    llama_trainer.train(resume_from_checkpoint=llama_ckpt)

    # Save final adapter
    print("\nSaving Llama final adapter...")
    llama_trainer.model.save_pretrained(LLAMA_OUT_DIR)
    llama_tok.save_pretrained(LLAMA_OUT_DIR)
    wandb.finish()

    llama_metrics = llama_trainer.evaluate()
    print(f"\n✅ Llama v1 Done! Eval Loss: {llama_metrics.get('eval_loss','N/A'):.4f}")

    del llama_model, llama_trainer
    torch.cuda.empty_cache(); gc.collect()
    print("🧹 GPU cleared.")

## 8. Day 4 Summary & Metadata Save

In [ ]:
def dir_size_mb(path):
    if not os.path.exists(path): return 0
    total = sum(os.path.getsize(os.path.join(r,f))
                for r,_,files in os.walk(path) for f in files)
    return round(total/(1024**2), 1)

metadata = {
    "Day": "Day 4 - Running v1 Training",
    "Jira_Task": "KAN-26",
    "Engineer": "Deepana Nirmal",
    "Domain": "SME Daily Business",
    "Models_Trained": {
        "qwen_sme_v1": {
            "base_model": "Qwen/Qwen2.5-7B-Instruct",
            "adapter_path": QWEN_OUT_DIR,
            "adapter_size_mb": dir_size_mb(QWEN_OUT_DIR),
            "eval_loss": round(qwen_metrics.get('eval_loss', 0), 4)
        },
        "llama_sme_v1": {
            "base_model": "meta-llama/Meta-Llama-3-8B-Instruct",
            "adapter_path": LLAMA_OUT_DIR,
            "adapter_size_mb": dir_size_mb(LLAMA_OUT_DIR),
            "eval_loss": round(llama_metrics.get('eval_loss', 0), 4)
        }
    },
    "Training_Config": {
        "dataset": "train_v1.json",
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "epochs": 3,
        "effective_batch_size": 16,
        "learning_rate": 2e-4,
        "scheduler": "cosine",
        "quantization": "4-bit NF4 double quant",
        "lora_r": 16, "lora_alpha": 32,
        "optimizer": "paged_adamw_32bit",
        "fp16": False, "bf16": False,
        "wandb_project": WANDB_PROJECT,
        "checkpoint_resume": True
    },
    "Status": "V1_ADAPTERS_TRAINED_AND_SAVED"
}

meta_path = os.path.join(PROJECT_ROOT, 'day4_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("✅ Day 4 (KAN-26) Complete!")
print(json.dumps(metadata, indent=2))
print("\n🎉 Ready for Day 5: First Inference & ROUGE/BLEU Evaluation (KAN-30)!")